# 10 — Generalization benchmark: 2017–2023 → 2024 (latest metrics)

Loads your benchmark run directory and recomputes AUROC/AUPRC from predictions so you never rely on outdated figures.


In [7]:
from __future__ import annotations

import os, sys, subprocess, json, re
from pathlib import Path
from datetime import datetime

REPO_ROOT = Path.cwd().parent
print("REPO_ROOT:", REPO_ROOT)

def sh(cmd: str, check: bool=True) -> None:
    """Run a shell command (prints it first)."""
    print("\n▶", cmd)
    subprocess.run(cmd, shell=True, check=check, cwd=REPO_ROOT)


def pick_first_existing(*cands: str) -> Path:
    for c in cands:
        p = REPO_ROOT / c
        if p.exists():
            return p
    return Path(cands[0])

def require_exists(p: Path, what: str="path") -> Path:
    if not p.exists():
        raise FileNotFoundError(f"Missing {what}: {p}")
    return p

def newest_path(glob_pat: str) -> Path | None:
    paths = list(REPO_ROOT.glob(glob_pat))
    if not paths:
        return None
    paths.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return paths[0]

def show_tree(root: Path, max_lines: int=200) -> None:
    i = 0
    for p in sorted(root.rglob("*")):
        if i >= max_lines:
            print("... (truncated)")
            return
        if p.is_dir():
            continue
        rel = p.relative_to(root)
        print(rel)
        i += 1

RUN_DIR = REPO_ROOT / "runs/eval/benchmark/labeled_bench_time_2017_2023_vs_2024_diag" 
require_exists(RUN_DIR, "benchmark run dir")
print("RUN_DIR:", RUN_DIR)
show_tree(RUN_DIR, max_lines=200)


REPO_ROOT: /Users/ameerfiras/REDNET-ML
RUN_DIR: /Users/ameerfiras/REDNET-ML/runs/eval/benchmark/labeled_bench_time_2017_2023_vs_2024_diag
README.txt
bootstrap_ci_per_seed.json
delong_vs_main.csv
mcnemar_vs_main.csv
per_seed_metrics.csv
summary_mean_std.csv


## 10.1 Report AUROC/AUPRC


In [8]:
import pandas as pd
from pathlib import Path

# RUN_DIR should already be REPO_ROOT-anchored earlier
per_seed_path = RUN_DIR / "per_seed_metrics.csv"
summary_path  = RUN_DIR / "summary_mean_std.csv"

if per_seed_path.exists():
    per_seed = pd.read_csv(per_seed_path)
    display(per_seed.head(10))
    print("per_seed rows:", len(per_seed), "cols:", len(per_seed.columns))
else:
    print("Missing:", per_seed_path)

if summary_path.exists():
    summary = pd.read_csv(summary_path)
    display(summary.head(50))
    print("summary rows:", len(summary), "cols:", len(summary.columns))
else:
    print("Missing:", summary_path)

# Convenience: show best models by roc_auc_mean (if present)
if summary_path.exists() and "roc_auc_mean" in summary.columns:
    best = summary.sort_values("roc_auc_mean", ascending=False)
    display(best.head(20))


,seed,model,n_train,n_test,pos_rate_test,roc_auc,pr_auc,precision,recall,f1,accuracy,kappa,thr_bestf1
0,0,hab_prob,1980,279,0.050179,0.986523,0.728578,0.666667,0.714286,0.689655,0.967742,0.672663,0.614458
1,0,p_frcnn_r50_med,1980,279,0.050179,0.556873,0.058603,0.000000,0.000000,0.000000,0.924731,-0.034611,0.971888
2,0,p_frcnn_mb_med,1980,279,0.050179,0.434771,0.056048,0.000000,0.000000,0.000000,0.939068,-0.018030,0.373494
3,0,p_ssd_mb_med,1980,279,0.050179,0.508895,0.072408,0.048649,0.642857,0.090452,0.351254,-0.003139,0.417671
4,0,index::fai_mean,1980,279,0.050179,0.590836,0.229136,0.057143,0.714286,0.105820,0.394265,0.014217,0.389558
5,0,rf_baseline,1980,279,0.050179,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.060241
6,1,hab_prob,1980,279,0.050179,0.986523,0.728578,0.666667,0.714286,0.689655,0.967742,0.672663,0.614458
7,1,p_frcnn_r50_med,1980,279,0.050179,0.556873,0.058603,0.000000,0.000000,0.000000,0.924731,-0.034611,0.971888
8,1,p_frcnn_mb_med,1980,279,0.050179,0.434771,0.056048,0.000000,0.000000,0.000000,0.939068,-0.018030,0.373494
9,1,p_ssd_mb_med,1980,279,0.050179,0.508895,0.072408,0.048649,0.642857,0.090452,0.351254,-0.003139,0.417671


per_seed rows: 60 cols: 13


,model,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,f1_mean,f1_std,accuracy_mean,kappa_mean,precision_mean,recall_mean
0,hab_prob,0.986523,0.0,0.728578,0.000000e+00,0.689655,0.0,0.967742,0.672663,0.666667,0.714286
1,index::fai_mean,0.590836,0.0,0.229136,0.000000e+00,0.105820,0.0,0.394265,0.014217,0.057143,0.714286
2,p_frcnn_mb_med,0.434771,0.0,0.056048,0.000000e+00,0.000000,0.0,0.939068,-0.018030,0.000000,0.000000
3,p_frcnn_r50_med,0.556873,0.0,0.058603,0.000000e+00,0.000000,0.0,0.924731,-0.034611,0.000000,0.000000
4,p_ssd_mb_med,0.508895,0.0,0.072408,0.000000e+00,0.090452,0.0,0.351254,-0.003139,0.048649,0.642857
5,rf_baseline,1.000000,0.0,1.000000,7.401487e-17,1.000000,0.0,1.000000,1.000000,1.000000,1.000000


summary rows: 6 cols: 11


,model,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,f1_mean,f1_std,accuracy_mean,kappa_mean,precision_mean,recall_mean
5,rf_baseline,1.000000,0.0,1.000000,7.401487e-17,1.000000,0.0,1.000000,1.000000,1.000000,1.000000
0,hab_prob,0.986523,0.0,0.728578,0.000000e+00,0.689655,0.0,0.967742,0.672663,0.666667,0.714286
1,index::fai_mean,0.590836,0.0,0.229136,0.000000e+00,0.105820,0.0,0.394265,0.014217,0.057143,0.714286
3,p_frcnn_r50_med,0.556873,0.0,0.058603,0.000000e+00,0.000000,0.0,0.924731,-0.034611,0.000000,0.000000
4,p_ssd_mb_med,0.508895,0.0,0.072408,0.000000e+00,0.090452,0.0,0.351254,-0.003139,0.048649,0.642857
2,p_frcnn_mb_med,0.434771,0.0,0.056048,0.000000e+00,0.000000,0.0,0.939068,-0.018030,0.000000,0.000000


## 10.2 Benchmark script help / rerun


In [9]:

p = REPO_ROOT / "scripts/eval/rigorous_labeled_bench.py"
print("Exists:", p.exists())
if p.exists():
    sh("python scripts/eval/rigorous_labeled_bench.py --help", check=False)

# Example:
# sh("python scripts/eval/rigorous_labeled_bench.py --out_dir runs/eval/benchmark/labeled_bench_time_2017_2023_vs_2024_diag")


Exists: True

▶ python scripts/eval/rigorous_labeled_bench.py --help
usage: Rigorous labeled benchmark: baselines + seeds + CI + significance + diagnostics
       [-h] --labeled_glob LABELED_GLOB --outdir OUTDIR
       [--label_col LABEL_COL] [--score_cols SCORE_COLS [SCORE_COLS ...]]
       [--index_col INDEX_COL] [--seeds SEEDS]
       [--train_end_year TRAIN_END_YEAR] [--test_year TEST_YEAR]
       [--test_size TEST_SIZE] [--trusted_only] [--rf_mode {auto,safe}]
       [--rf_drop [RF_DROP ...]] [--rf_topk RF_TOPK]
       [--split_mode {time,group,rolling_year}]
       [--rolling_start_year ROLLING_START_YEAR]
       [--rolling_end_year ROLLING_END_YEAR] [--thr_policy {best_f1}]
       [--n_boot N_BOOT]

options:
  -h, --help            show this help message and exit
  --labeled_glob LABELED_GLOB
  --outdir OUTDIR
  --label_col LABEL_COL
  --score_cols SCORE_COLS [SCORE_COLS ...]
  --index_col INDEX_COL
  --seeds SEEDS
  --train_end_year TRAIN_END_YEAR
  --test_year TEST_YEAR
  --te